# 03 — Classification: predict next-observation direction

This notebook consolidates the classification experiments from the final notebook. It compares Random Forest, SVM-RBF and MLP using three feature sets (7, 15-paper, 15-stationary) on validation data. Each model's feature set for the held-out test is chosen from validation results only.

**Important:** test results must not be used to choose a model or tune its settings.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fx_utils import PAIRS, PROCESSED_DIR, RESULTS_DIR, chronological_split, get_feature_sets

data = {
    pair: pd.read_csv(PROCESSED_DIR / f"{pair}_model_data.csv", index_col=0, parse_dates=True)
    for pair in PAIRS
}
splits = {pair: chronological_split(data[pair]) for pair in PAIRS}

def make_classifier(name):
    if name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=1000, max_depth=100, random_state=42, n_jobs=-1
        )
    if name == "SVM-RBF":
        return make_pipeline(StandardScaler(), SVC(kernel="rbf"))
    if name == "MLP":
        return make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(64, 32), max_iter=500,
                early_stopping=True, random_state=42
            )
        )
    raise ValueError(f"Unknown model: {name}")

In [2]:
# Validation experiments: 3 models × 3 feature sets × 3 currency pairs
validation_rows = []

for pair in PAIRS:
    train, val = splits[pair]["train"], splits[pair]["val"]
    target_col = f"{pair}_target_direction"

    for feature_set, feature_cols in get_feature_sets(pair).items():
        for model_name in ["Random Forest", "SVM-RBF", "MLP"]:
            model = make_classifier(model_name)
            model.fit(train[feature_cols], train[target_col].astype(int))
            predictions = model.predict(val[feature_cols])
            validation_rows.append({
                "Currency Pair": pair,
                "Model": model_name,
                "Features": feature_set,
                "Validation Accuracy": accuracy_score(val[target_col].astype(int), predictions),
                "Balanced Accuracy": balanced_accuracy_score(val[target_col].astype(int), predictions)
            })

validation_results = pd.DataFrame(validation_rows)
display(validation_results.round(4))

,Currency Pair,Model,Features,Validation Accuracy,Balanced Accuracy
0,GBP_USD,Random Forest,7,0.4901,0.4921
1,GBP_USD,SVM-RBF,7,0.5298,0.5255
2,GBP_USD,MLP,7,0.4864,0.4881
3,GBP_USD,Random Forest,15-paper,0.5353,0.5113
4,GBP_USD,SVM-RBF,15-paper,0.5298,0.5225
5,GBP_USD,MLP,15-paper,0.4901,0.4848
6,GBP_USD,Random Forest,15-stationary,0.5045,0.5035
7,GBP_USD,SVM-RBF,15-stationary,0.5027,0.5022
8,GBP_USD,MLP,15-stationary,0.4919,0.4953
9,USD_EUR,Random Forest,7,0.5009,0.5027


In [3]:
# Average validation scores across the three currency pairs
average_validation = (
    validation_results
    .groupby(["Model", "Features"])[["Validation Accuracy", "Balanced Accuracy"]]
    .mean()
    .reset_index()
)
display(average_validation.round(4))

,Model,Features,Validation Accuracy,Balanced Accuracy
0,MLP,15-paper,0.5316,0.5109
1,MLP,15-stationary,0.5033,0.5044
2,MLP,7,0.5009,0.4980
3,Random Forest,15-paper,0.5377,0.5071
4,Random Forest,15-stationary,0.4846,0.4842
5,Random Forest,7,0.4943,0.4928
6,SVM-RBF,15-paper,0.5497,0.5193
7,SVM-RBF,15-stationary,0.4943,0.4942
8,SVM-RBF,7,0.5190,0.5141


In [4]:
# Final test evaluation: Random Forest and SVM-RBF vs majority baseline.
# Each model uses the feature set with the highest mean validation balanced accuracy.
# Refit on train + validation after model selection, then evaluate test once.
selected_features = (
    average_validation.sort_values("Balanced Accuracy", ascending=False)
    .drop_duplicates("Model").set_index("Model")["Features"].to_dict()
)
print("Feature set chosen on validation:",
      {m: selected_features[m] for m in ["Random Forest", "SVM-RBF"]})

test_rows = []
test_reports = []
test_prediction_records = []

for pair in PAIRS:
    train, val, test = splits[pair]["train"], splits[pair]["val"], splits[pair]["test"]
    development = pd.concat([train, val])
    target_col = f"{pair}_target_direction"
    feature_sets = get_feature_sets(pair)
    y_dev = development[target_col].astype(int)
    y_test = test[target_col].astype(int)

    baseline = DummyClassifier(strategy="most_frequent").fit(development[feature_sets["7"]], y_dev)
    baseline_pred = baseline.predict(test[feature_sets["7"]])
    test_rows.append({
        "Currency Pair": pair, "Model": "Majority baseline", "Features": "none",
        "Test Accuracy": accuracy_score(y_test, baseline_pred),
        "Test Balanced Accuracy": balanced_accuracy_score(y_test, baseline_pred)
    })

    for model_name in ["Random Forest", "SVM-RBF"]:
        feature_cols = feature_sets[selected_features[model_name]]
        model = make_classifier(model_name)
        model.fit(development[feature_cols], y_dev)
        predictions = model.predict(test[feature_cols])
        test_rows.append({
            "Currency Pair": pair, "Model": model_name, "Features": selected_features[model_name],
            "Test Accuracy": accuracy_score(y_test, predictions),
            "Test Balanced Accuracy": balanced_accuracy_score(y_test, predictions)
        })
        test_reports.append((pair, model_name, classification_report(
            y_test, predictions, zero_division=0
        )))

        # Export out-of-sample predictions for the selected Random Forest model
        if model_name == "Random Forest":
            test_prediction_records.append(pd.DataFrame({
                "date": test.index,
                "pair": pair,
                "y_true": y_test.values,
                "y_pred": predictions
            }))

test_results = pd.DataFrame(test_rows)
display(test_results.round(4))

for pair, model_name, report in test_reports:
    print(f"\n{pair} — {model_name}\n{report}")

Feature set chosen on validation: {'Random Forest': '15-paper', 'SVM-RBF': '15-paper'}


,Currency Pair,Model,Features,Test Accuracy,Test Balanced Accuracy
0,GBP_USD,Majority baseline,none,0.5045,0.5000
1,GBP_USD,Random Forest,15-paper,0.5244,0.5254
2,GBP_USD,SVM-RBF,15-paper,0.5118,0.5133
3,USD_EUR,Majority baseline,none,0.4756,0.5000
4,USD_EUR,Random Forest,15-paper,0.5262,0.5309
5,USD_EUR,SVM-RBF,15-paper,0.5099,0.5223
6,USD_JPY,Majority baseline,none,0.5570,0.5000
7,USD_JPY,Random Forest,15-paper,0.5118,0.4953
8,USD_JPY,SVM-RBF,15-paper,0.5353,0.4843



GBP_USD — Random Forest
              precision    recall  f1-score   support

           0       0.54      0.41      0.47       279
           1       0.52      0.64      0.57       274

    accuracy                           0.52       553
   macro avg       0.53      0.53      0.52       553
weighted avg       0.53      0.52      0.52       553


GBP_USD — SVM-RBF
              precision    recall  f1-score   support

           0       0.53      0.34      0.41       279
           1       0.51      0.69      0.58       274

    accuracy                           0.51       553
   macro avg       0.52      0.51      0.50       553
weighted avg       0.52      0.51      0.50       553


USD_EUR — Random Forest
              precision    recall  f1-score   support

           0       0.50      0.63      0.56       263
           1       0.56      0.43      0.49       290

    accuracy                           0.53       553
   macro avg       0.53      0.53      0.52       553
weigh

In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
validation_results.to_csv(RESULTS_DIR / "classification_validation_results.csv", index=False)
average_validation.to_csv(RESULTS_DIR / "classification_validation_averages.csv", index=False)
test_results.to_csv(RESULTS_DIR / "classification_test_results.csv", index=False)

# Export combined out-of-sample test predictions for Notebook 07
test_predictions_df = pd.concat(test_prediction_records, ignore_index=True)
test_predictions_path = RESULTS_DIR / "classification_test_predictions.csv"
test_predictions_df.to_csv(test_predictions_path, index=False)

print("Saved classification summary tables to:", RESULTS_DIR)
print("\nSaved test predictions to:", test_predictions_path)
print("  Shape:", test_predictions_df.shape)
print("  Columns:", test_predictions_df.columns.tolist())
print(f"  Date range: {test_predictions_df['date'].min().date()} to {test_predictions_df['date'].max().date()}")
print("\nPrediction count per pair:")
print(test_predictions_df["pair"].value_counts().to_string())


Saved classification summary tables to: D:\NeuDocuments\SideProject\MiniProject\ML\Foreign-exchange-forecast\results

Saved test predictions to: D:\NeuDocuments\SideProject\MiniProject\ML\Foreign-exchange-forecast\results\classification_test_predictions.csv
  Shape: (1659, 4)
  Columns: ['date', 'pair', 'y_true', 'y_pred']
  Date range: 2024-08-27 to 2026-10-08

Prediction count per pair:
pair
GBP_USD    553
USD_EUR    553
USD_JPY    553


In [6]:
# Validation cell: Reload saved test predictions and verify schema, integrity, and coverage
reloaded = pd.read_csv(test_predictions_path, parse_dates=["date"])

required_cols = ["date", "pair", "y_true", "y_pred"]
assert list(reloaded.columns) == required_cols, f"Schema mismatch: {reloaded.columns.tolist()} vs {required_cols}"
assert reloaded[required_cols].isna().sum().sum() == 0, f"Found missing values: {reloaded.isna().sum().to_dict()}"
assert not reloaded.duplicated(subset=["date", "pair"]).any(), "Found duplicate (date, pair) rows!"
assert set(reloaded["pair"].unique()) == set(PAIRS), f"Pair coverage mismatch: {reloaded['pair'].unique()}"

for p in PAIRS:
    subset = reloaded[reloaded["pair"] == p]
    assert len(subset) == len(splits[p]["test"]), f"Row count mismatch for {p}: {len(subset)} vs {len(splits[p]['test'])}"
    assert subset["date"].is_monotonic_increasing, f"Dates not sorted chronologically for {p}"

print("="*60)
print("VALIDATION PASSED: classification_test_predictions.csv")
print("="*60)
display(reloaded.groupby("pair").agg(
    total_observations=("date", "count"),
    start_date=("date", lambda s: s.min().date()),
    end_date=("date", lambda s: s.max().date()),
    accuracy=("y_true", lambda y: (y == reloaded.loc[y.index, "y_pred"]).mean())
))


VALIDATION PASSED: classification_test_predictions.csv


,total_observations,start_date,end_date,accuracy
pair,,,,
GBP_USD,553,2024-08-27,2026-10-08,0.524412
USD_EUR,553,2024-08-27,2026-10-08,0.526221
USD_JPY,553,2024-08-27,2026-10-08,0.511754


## How to report results

- Validation results are used for model comparison; test results estimate performance on later unseen observations.
- Compare models with the majority-class baseline and report both accuracy and balanced accuracy.
- Results near 50% indicate limited directional signal. Accuracy alone does not establish trading profitability.
- The data source and time period differ from the original paper, so comparisons are indicative rather than exact replications.